## Hello, Data!

In [15]:
import pandas as pd

# Load the raw sales data
df1 = pd.read_csv('../data/5000_Sales_Records.csv')

# Display the first 3 rows
df1.head(3)

,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Central America and the Caribbean,Antigua and Barbuda,Baby Food,Online,M,12/20/2013,957081544,1/11/2014,552,255.28,159.42,140914.56,87999.84,52914.72
1,Central America and the Caribbean,Panama,Snacks,Offline,C,7/5/2010,301644504,7/26/2010,2167,152.58,97.44,330640.86,211152.48,119488.38
2,Europe,Czech Republic,Beverages,Offline,C,9/12/2011,478051030,9/29/2011,4778,47.45,31.79,226716.10,151892.62,74823.48


### Missing values
The above dataset doesn't contain required columns like 'shipping_city' and 'coupon_code'. Therefore I will create my own synthetic dataset to fulfill all the requirements

In [ ]:
import random
import pandas as pd

# Generate 500 rows of sample data matching lab requirements
data = {
    "date": pd.date_range(start="2026-01-01", periods=500, freq="h"),
    "customer_id": [
        f"CUST_{random.randint(1000, 1100)}" for _ in range(500)
    ],
    "product": random.choices(
        ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones"], k=500
    ),
    "price": [round(random.uniform(20.0, 1200.0), 2) for _ in range(500)],
    "quantity": [random.randint(1, 5) for _ in range(500)],
    "coupon_code": random.choices(
        ["SAVE10", "WELCOME20", "FREESHIP", "NONE"], k=500
    ),
    "shipping_city": random.choices(
        ["Toronto", "Vancouver", "Montreal", "Calgary", "Ottawa"], k=500
    ),
}

df = pd.DataFrame(data)

# Save it to your data/ folder
df.to_csv("../data/sales_500.csv", index=False)
print("Dataset created successfully with all required columns!")

Dataset created successfully with all required columns!


## Hello, Data! Again 

In [17]:
import pandas as pd

# Load the raw sales data
df = pd.read_csv('../data/sales_500.csv')

# Display the first 3 rows
df.head(3)

,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,2026-01-01 00:00:00,CUST_1085,Headphones,1186.10,4,SAVE10,Toronto
1,2026-01-01 01:00:00,CUST_1012,Laptop,1162.77,5,NONE,Toronto
2,2026-01-01 02:00:00,CUST_1004,Keyboard,112.47,2,FREESHIP,Montreal


## Pick the Right Container

**What We do :**
We use dictionaries when we need fast lookups by key (like finding a customer by customer_id), and sets when we only care about unique items without duplicates (like finding a list of unique shipping cities). In contrast, we use namedtuples when we want a lightweight, immutable data structure to hold a single fixed record (like a single sales transaction) with named fields instead of regular numeric indexes.

## Implement Functions and Data Structure

**What to do:** Build a small Python class that represents a single transaction item, including a method to clean data and calculate totals.

In [4]:
class SaleTransaction:
    def __init__(self, customer_id, product, price, quantity, shipping_city):
        self.customer_id = customer_id
        self.product = product
        self.price = float(price)
        self.quantity = int(quantity)
        self.shipping_city = shipping_city.strip().title()

    def total(self):
        return self.price * self.quantity

# Quick test of our class
sample = SaleTransaction("C101", " Widget ", 15.5, 2, " new york ")
print(f"Clean City: {sample.shipping_city}, Total: ${sample.total()}")

Clean City: New York, Total: $31.0


## Bulk Loaded

**What to do:** Take the rows from Pandas DataFrame and turn them into a list of dictionaries (or objects)

In [18]:
# Convert DataFrame rows into a list of dictionaries
transactions_dict = df.to_dict(orient='records')

# Inspect the first converted item
print(transactions_dict[0])

{'date': '2026-01-01 00:00:00', 'customer_id': 'CUST_1085', 'product': 'Headphones', 'price': 1186.1, 'quantity': 4, 'coupon_code': 'SAVE10', 'shipping_city': 'Toronto'}


## Quick Profiling

**What to do:** Find basic stats—minimum price, average price, maximum price, and count of unique cities using sets.

In [20]:
# Calculate price stats using pandas or standard python
# Calculate price statistics
min_price = df['price'].min()
max_price = df['price'].max()
mean_price = df['price'].mean()

# Use a SET to get unique countries
unique_cities = set(df['shipping_city'].dropna())

print(f"Price Stats - Minimum Price: ${min_price:.2f}, Mean: ${mean_price:.2f}, Maximum Price: ${max_price:.2f}")
print(f"Unique Cities Count: {len(unique_cities)}")


Price Stats - Minimum Price: $20.37, Mean: $619.91, Maximum Price: $1198.40
Unique Cities Count: 5


## Spot the Grime

**What to do:** Look closely at data and identify 3 dirty data problems

**Example :** 

 * Missing values in coupon_code.

* Messy text formatting with extra spaces in shipping_city.

* Negative or zero values in quantity or price.

## Cleaning Rules

In [21]:
initial_count = len(df)

# Fix missing coupon codes with a default value
df['coupon_code'] = df['coupon_code'].fillna('NONE')

# Strip extra spaces and title-case cities
df['shipping_city'] = df['shipping_city'].astype(str).str.strip().str.title()

# Filter out bad quantities (keep only positive values)
df_clean = df[df['quantity'] > 0].copy()

final_count = len(df_clean)
print(f"Rows before cleaning: {initial_count} | Rows after cleaning: {final_count}")

Rows before cleaning: 500 | Rows after cleaning: 500


## Transformations

**What to do:** Convert messy or text fields into useful formats (e.g., converting a coupon code string into an actual numeric percentage discount).

In [26]:
# Convert coupon code into a numeric discount multiplier
def parse_coupon(code):
    if code == 'SAVE10':
        return 0.10
    elif code == 'WELCOME20':
        return 0.20
    return 0.00

df_clean['discount_rate'] = df_clean['coupon_code'].apply(parse_coupon)
df_clean.head(4)

,date,customer_id,product,price,quantity,coupon_code,shipping_city,discount_rate,days_since_purchase,net_revenue
0,2026-01-01 00:00:00,CUST_1085,Headphones,1186.10,4,SAVE10,Toronto,0.1,264,4269.96
1,2026-01-01 01:00:00,CUST_1012,Laptop,1162.77,5,NONE,Toronto,0.0,264,5813.85
2,2026-01-01 02:00:00,CUST_1004,Keyboard,112.47,2,FREESHIP,Montreal,0.0,264,224.94
3,2026-01-01 03:00:00,CUST_1019,Monitor,418.89,1,FREESHIP,Ottawa,0.0,264,418.89


## Feature Engineering

**What to do:** Create a brand new useful column from existing data (e.g., days_since_purchase or total_spend).

In [25]:
from datetime import datetime

# Convert date column to datetime type
df_clean['date'] = pd.to_datetime(df_clean['date'])

# Calculate days since purchase relative to today
today = pd.to_datetime('today')
df_clean['days_since_purchase'] = (today - df_clean['date']).dt.days

# Calculate final revenue column
df_clean['net_revenue'] = (df_clean['price'] * df_clean['quantity']) * (1 - df_clean['discount_rate'])
df_clean.head(4)

,date,customer_id,product,price,quantity,coupon_code,shipping_city,discount_rate,days_since_purchase,net_revenue
0,2026-01-01 00:00:00,CUST_1085,Headphones,1186.10,4,SAVE10,Toronto,0.1,264,4269.96
1,2026-01-01 01:00:00,CUST_1012,Laptop,1162.77,5,NONE,Toronto,0.0,264,5813.85
2,2026-01-01 02:00:00,CUST_1004,Keyboard,112.47,2,FREESHIP,Montreal,0.0,264,224.94
3,2026-01-01 03:00:00,CUST_1019,Monitor,418.89,1,FREESHIP,Ottawa,0.0,264,418.89


## Mini-Aggregation


**What to do:** Group data to summarize key insights (e.g., total sales revenue grouped by city).

In [28]:
# Group by shipping city and calculate total net revenue
city_revenue = df_clean.groupby('shipping_city')['net_revenue'].sum().reset_index()
city_revenue = city_revenue.sort_values(by='net_revenue', ascending=False).reset_index(drop=True)

city_revenue.head(5)

,shipping_city,net_revenue
0,Ottawa,191317.387
1,Calgary,181360.182
2,Montreal,181018.093
3,Toronto,162407.098
4,Vancouver,157531.459


## Serialization Checkpoint

**What to do:** Save clean, processed dataset to a JSON file.

In [30]:
# Save clean dataset as JSON
df_clean.to_json('../src/clean_sales_data.json', orient='records', indent=2)
print("Clean data successfully saved to JSON!")

Clean data successfully saved to JSON!


C:\Users\shain\AppData\Local\Temp\ipykernel_19180\1603637289.py:2: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_clean.to_json('../src/clean_sales_data.json', orient='records', indent=2)


## Soft Interview Reflection

**What to do:** Write a short Markdown response (< 120 words) explaining why functions were helpful.

Using functions and classes made completing this lab so much easier! Previously, if we had to clean raw data or apply specific rules to every single row, the code would quickly duplicate and become impossible to manage.

By designing a reusable clean() function and a clean class structure, I kept my code modular, readable, and neat. If the dataset grows or new columns are added in the future, I don’t have to rewrite everything from scratch—I can just update that specific function. This approach speeds up debugging and helps follow industry standards when writing production code.

## Data-Dictionary Section

Below is the merged data dictionary combining the fields from our primary synthetic transactions file (`sales_500.csv`) and our secondary metadata lookup file (`city_info.csv`).

| Field | Type | Description | Source | Creation / Mapping Method |
| :--- | :--- | :--- | :--- | :--- |
| `date` | Timestamp | Date and hour of the transaction | `sales_500.csv` (Primary) | Generated via Python pandas date range |
| `customer_id` | String | Unique identifier for the customer | `sales_500.csv` (Primary) | Synthetic random generation (`CUST_1000+`) |
| `product` | String | Name of the purchased item | `sales_500.csv` (Primary) | Random choice from predefined product list |
| `price` | Float | Unit price of the product | `sales_500.csv` (Primary) | Random uniform float rounded to 2 decimals |
| `quantity` | Integer | Units bought in the transaction | `sales_500.csv` (Primary) | Random integer between 1 and 5 |
| `coupon_code` | String | Promo or discount code applied | `sales_500.csv` (Primary) | Randomly assigned promotional strings or `NONE` |
| `shipping_city` | String | Destination city for the order | `sales_500.csv` & `city_info.csv` | Mapped and enriched using the secondary city lookup file |
| `city_population` | Integer | Population metadata for the destination city | `city_info.csv` (Secondary Metadata) | Merged into the main dataframe via `shipping_city` lookup |